# TASK-011: full Berton Case-0 long-integration classification

Notebook-first workflow for classifying the canonical oscillatory Case-0 trajectory after the TASK-010 `z_W0` equilibrium Hopf screen did not identify a Hopf candidate. The executable helper keeps the long integration and artifact generation reproducible while this notebook records the exact command, settings, outputs, and interpretation.

## Canonical configuration

- `atmosphere_for_case(0, oscillatory=True)` (`z_W0=9 km`, `W_a0=0.6 m/s`, `H_a3=0.61`, `eta_override=None`)
- `initial_state_for_case(0)`
- `SimulationConfig(include_coriolis=False, stop_on_nonpositive_mass=False)`
- SciPy `BDF` and `LSODA`; no explicit-Euler long integration is used for classification
- Required horizon: 200 h; extension rule: if 150-200 h envelope remains nonzero/ambiguous, extend both solvers to 500 h and classify on 450-500 h

In [1]:
from pathlib import Path
import sys

repo = Path.cwd()
if not (repo / 'pyproject.toml').exists():
    repo = Path.cwd().parents[2]
episode = repo / 'episodes' / '05-full-model-oscillatory-orbit'
script_dir = episode / 'scripts'
sys.path.insert(0, str(script_dir))

import berton_full_task011_classify as task011

task011.OUT_DIR

PosixPath('/home/iross/research/ongoing/ai-class/code/cloud-rom/episodes/05-full-model-oscillatory-orbit/outputs/task011')

In [2]:
# Reproduce all curated outputs. Equivalent command from repository root:
# uv run python episodes/05-full-model-oscillatory-orbit/scripts/berton_full_task011_classify.py
task011.main()

Wrote TASK-011 outputs to /home/iross/research/ongoing/ai-class/code/cloud-rom/episodes/05-full-model-oscillatory-orbit/outputs/task011
method  duration_h  base_duration_h                                                                                                         extension_rule   z_final_m  u_final_m_s  w_final_m_s   m_final_kg  z_amp_150_200h_m  z_amp_450_500h_m  amp_ratio_late_to_base  late_peak_period_h          classification
   BDF       500.0            200.0 Amplitude at 150-200 h remained nonzero; extend both BDF and LSODA to 500 h and classify using the 450-500 h envelope. 9618.027979      1.90986 2.279986e-07 1.080229e-09         19.208274          0.009236                0.000481           10.203704 damped/equilibrium-like
 LSODA       500.0            200.0 Amplitude at 150-200 h remained nonzero; extend both BDF and LSODA to 500 h and classify using the 450-500 h envelope. 9618.027988      1.90986 2.263444e-07 1.080229e-09         19.208205          0.009209   

In [3]:
import pandas as pd
out = task011.OUT_DIR
summary = pd.read_csv(out / 'summary.csv')
solver = pd.read_csv(out / 'solver_agreement.csv')
equilibrium = pd.read_csv(out / 'equilibrium.csv')
eigenvalues = pd.read_csv(out / 'eigenvalues.csv')
summary

,method,duration_h,base_duration_h,extension_rule,z_final_m,u_final_m_s,w_final_m_s,m_final_kg,z_amp_150_200h_m,z_amp_450_500h_m,amp_ratio_late_to_base,late_peak_period_h,classification
0,BDF,500.0,200.0,Amplitude at 150-200 h remained nonzero; exten...,9618.027979,1.90986,2.279986e-07,1.080229e-09,19.208274,0.009236,0.000481,10.203704,damped/equilibrium-like
1,LSODA,500.0,200.0,Amplitude at 150-200 h remained nonzero; exten...,9618.027988,1.90986,2.263444e-07,1.080229e-09,19.208205,0.009209,0.000479,10.194444,damped/equilibrium-like


In [4]:
solver

,horizon_h,max_abs_z_diff_m,max_abs_w_diff_m_s,late_max_abs_z_diff_m,late_max_abs_w_diff_m_s
0,200.0,0.00649,0.000007,0.001637,2.996330e-07
1,500.0,0.00649,0.000007,0.000044,7.942544e-09


In [5]:
equilibrium, eigenvalues

(   root_success             root_message       z_eq_m  u_eq_m_s      w_eq_m_s  \
 0          True  The solution converged.  9618.027532  1.909862 -2.908454e-21   
 
         m_eq_kg      rhs_norm     rhs_z_m_s    rhs_u_m_s2    rhs_w_m_s2  \
 0  1.080229e-09  2.131937e-13 -2.908454e-21  3.628674e-15 -2.131628e-13   
 
      rhs_m_kg_s  
 0 -3.795523e-25  ,
    eigenvalue_index  real_s_inv  imag_s_inv  period_h_if_complex  e_folding_h
 0                 1  -16.342094    0.000000                  NaN     0.000017
 1                 2  -18.718354    0.000000                  NaN     0.000015
 2                 3   -0.000007    0.000171            10.193947    39.344293
 3                 4   -0.000007   -0.000171            10.193947    39.344293)

## Verdict

The exported diagnostics classify the canonical no-Coriolis oscillatory Case-0 trajectory as **damped/equilibrium-like**. The 150-200 h altitude envelope remains nonzero, so the workflow extends both BDF and LSODA to 500 h. By 450-500 h, the envelope has collapsed by more than three orders of magnitude and the two solvers agree closely. The refined equilibrium seed and finite-difference eigenvalues include a stable complex pair with an approximately 10.19 h period and 39.34 h e-folding time, explaining the observed decaying oscillations without requiring a finite-amplitude limit cycle.